In [1]:
import pandas as pd
import numpy as np

import re
import string

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("manufacturing_instrumentation.csv")

df.head()

,timestamp,shift,line_id,machine_id,product_code,batch_id,operator_id,operator_experience,days_since_maintenance,maintenance_status,...,vibration_mm_s,line_speed_units_min,window_minutes,throughput_units,defect_rate_percent,scrap_kg,downtime_minutes,energy_kwh,ambient_temp_c,quality_alert
0,2025-10-25T20:07:55.475814,Day,L1,L1-M1,P-PEEK,B251025-882,OP-5,expert,16.4,ok,...,3.00,136.7,12,1640,1.94,25.61,3.08,60.97,21.8,False
1,2025-10-25T20:08:55.475814,Night,L4,L4-M1,P-NYLON,B251025-475,OP-1,expert,0.0,ok,...,3.19,126.6,8,1013,1.82,17.21,6.30,50.56,21.8,False
2,2025-10-25T21:01:55.475814,Day,L3,L3-M6,P-PEEK,B251025-270,OP-1,expert,2.5,ok,...,3.66,140.7,10,1406,0.48,7.71,4.76,55.21,27.1,False
3,2025-10-25T21:05:55.475814,Swing,L1,L1-M1,P-ABS,B251025-484,OP-2,intermediate,30.7,overdue,...,4.21,91.9,12,1102,1.89,13.17,9.25,50.51,25.0,False
4,2025-10-25T21:32:55.475814,Day,L3,L3-M6,P-NYLON,B251025-755,OP-3,novice,8.6,ok,...,2.40,85.5,11,940,2.16,15.67,3.17,48.89,24.7,False


In [3]:
df.info()
df.isnull().sum()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   timestamp               2000 non-null   str    
 1   shift                   2000 non-null   str    
 2   line_id                 2000 non-null   str    
 3   machine_id              2000 non-null   str    
 4   product_code            2000 non-null   str    
 5   batch_id                2000 non-null   str    
 6   operator_id             2000 non-null   str    
 7   operator_experience     2000 non-null   str    
 8   days_since_maintenance  2000 non-null   float64
 9   maintenance_status      2000 non-null   str    
 10  target_temp_c           2000 non-null   float64
 11  actual_temp_c           2000 non-null   float64
 12  humidity_percent        2000 non-null   float64
 13  pressure_bar            2000 non-null   float64
 14  vibration_mm_s          2000 non-null   float64
 15

timestamp                 0
shift                     0
line_id                   0
machine_id                0
product_code              0
batch_id                  0
operator_id               0
operator_experience       0
days_since_maintenance    0
maintenance_status        0
target_temp_c             0
actual_temp_c             0
humidity_percent          0
pressure_bar              0
vibration_mm_s            0
line_speed_units_min      0
window_minutes            0
throughput_units          0
defect_rate_percent       0
scrap_kg                  0
downtime_minutes          0
energy_kwh                0
ambient_temp_c            0
quality_alert             0
dtype: int64

In [7]:
df.select_dtypes(include=["object", "string"]).columns

Index(['timestamp', 'shift', 'line_id', 'machine_id', 'product_code',
       'batch_id', 'operator_id', 'operator_experience', 'maintenance_status'],
      dtype='str')

In [9]:
print(df.columns)

Index(['timestamp', 'shift', 'line_id', 'machine_id', 'product_code',
       'batch_id', 'operator_id', 'operator_experience',
       'days_since_maintenance', 'maintenance_status', 'target_temp_c',
       'actual_temp_c', 'humidity_percent', 'pressure_bar', 'vibration_mm_s',
       'line_speed_units_min', 'window_minutes', 'throughput_units',
       'defect_rate_percent', 'scrap_kg', 'downtime_minutes', 'energy_kwh',
       'ambient_temp_c', 'quality_alert'],
      dtype='str')


In [11]:

df.head()

,timestamp,shift,line_id,machine_id,product_code,batch_id,operator_id,operator_experience,days_since_maintenance,maintenance_status,...,vibration_mm_s,line_speed_units_min,window_minutes,throughput_units,defect_rate_percent,scrap_kg,downtime_minutes,energy_kwh,ambient_temp_c,quality_alert
0,2025-10-25T20:07:55.475814,Day,L1,L1-M1,P-PEEK,B251025-882,OP-5,expert,16.4,ok,...,3.00,136.7,12,1640,1.94,25.61,3.08,60.97,21.8,False
1,2025-10-25T20:08:55.475814,Night,L4,L4-M1,P-NYLON,B251025-475,OP-1,expert,0.0,ok,...,3.19,126.6,8,1013,1.82,17.21,6.30,50.56,21.8,False
2,2025-10-25T21:01:55.475814,Day,L3,L3-M6,P-PEEK,B251025-270,OP-1,expert,2.5,ok,...,3.66,140.7,10,1406,0.48,7.71,4.76,55.21,27.1,False
3,2025-10-25T21:05:55.475814,Swing,L1,L1-M1,P-ABS,B251025-484,OP-2,intermediate,30.7,overdue,...,4.21,91.9,12,1102,1.89,13.17,9.25,50.51,25.0,False
4,2025-10-25T21:32:55.475814,Day,L3,L3-M6,P-NYLON,B251025-755,OP-3,novice,8.6,ok,...,2.40,85.5,11,940,2.16,15.67,3.17,48.89,24.7,False


In [14]:
print(df.columns)


Index(['timestamp', 'shift', 'line_id', 'machine_id', 'product_code',
       'batch_id', 'operator_id', 'operator_experience',
       'days_since_maintenance', 'maintenance_status', 'target_temp_c',
       'actual_temp_c', 'humidity_percent', 'pressure_bar', 'vibration_mm_s',
       'line_speed_units_min', 'window_minutes', 'throughput_units',
       'defect_rate_percent', 'scrap_kg', 'downtime_minutes', 'energy_kwh',
       'ambient_temp_c', 'quality_alert'],
      dtype='str')


In [16]:
print(df.columns)

Index(['timestamp', 'shift', 'line_id', 'machine_id', 'product_code',
       'batch_id', 'operator_id', 'operator_experience',
       'days_since_maintenance', 'maintenance_status', 'target_temp_c',
       'actual_temp_c', 'humidity_percent', 'pressure_bar', 'vibration_mm_s',
       'line_speed_units_min', 'window_minutes', 'throughput_units',
       'defect_rate_percent', 'scrap_kg', 'downtime_minutes', 'energy_kwh',
       'ambient_temp_c', 'quality_alert'],
      dtype='str')


In [18]:
print(df.head())
print(df.columns.tolist())

                    timestamp  shift line_id machine_id product_code  \
0  2025-10-25T20:07:55.475814    Day      L1      L1-M1       P-PEEK   
1  2025-10-25T20:08:55.475814  Night      L4      L4-M1      P-NYLON   
2  2025-10-25T21:01:55.475814    Day      L3      L3-M6       P-PEEK   
3  2025-10-25T21:05:55.475814  Swing      L1      L1-M1        P-ABS   
4  2025-10-25T21:32:55.475814    Day      L3      L3-M6      P-NYLON   

      batch_id operator_id operator_experience  days_since_maintenance  \
0  B251025-882        OP-5              expert                    16.4   
1  B251025-475        OP-1              expert                     0.0   
2  B251025-270        OP-1              expert                     2.5   
3  B251025-484        OP-2        intermediate                    30.7   
4  B251025-755        OP-3              novice                     8.6   

  maintenance_status  ...  vibration_mm_s  line_speed_units_min  \
0                 ok  ...            3.00              

In [19]:
df.to_csv("manufacturing_clusters_output.csv", index=False)